In [7]:
pip install ipykernel requests python-frontmatter

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.4 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached idna-3.20-py3-none-any.whl (69 kB)
Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl (173 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [python-frontmatter]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'
resp = requests.get(url)

In [6]:
repository_data = []

# Create a ZipFile object from the downloaded content
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    if not (filename.endswith('.md') or filename.endswith('.mdx')):
        continue

    # Read and parse each file
    with zf.open(file_info) as f_in:
        content = f_in.read()
        post = frontmatter.loads(content)
        data = post.to_dict()
        data['filename'] = filename
        repository_data.append(data)

zf.close()

In [7]:
print(repository_data[1])

{'name': 'clear-backlog', 'description': 'Clear the FAQ repository backlog one item at a time: review and resolve all open pull requests first, then work through open issues. Use when the user asks to clear, review, process, or work through the FAQ PR/issue backlog. Check placement, duplicates, canonical sources, technical accuracy, and content quality; pause for approval on every item; and recommend issue eval coverage only when it protects a meaningful agent behavior.', 'content': '# Clear the FAQ Backlog\n\nProcess all open pull requests before starting the issue queue. Work strictly\none item at a time: investigate one item, show the user the original and the\nrecommendation, wait for approval, resolve it, and only then show the next item.\n\nDo not bulk-merge, bulk-close, or silently move past the approval checkpoint.\n\n## Prepare\n\n1. Read `AGENTS.md`, `CONTRIBUTING.md`, and the relevant course metadata.\n2. Check `git status --short --branch`. Preserve unrelated user changes.\

In [9]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md')
            or filename_lower.endswith('.mdx')):
            continue

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    zf.close()
    return repository_data

In [10]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1438
Evidently documents: 95
